# QACR — Colab runner

This notebook only **pulls the repo and runs scripts**. All logic lives in `src/` and `scripts/`.

**Every session:** `Runtime → Change runtime type → T4 GPU`, then run the cells top to bottom.

1. Edit the *Settings* cell (repo URL, and the script/config for the phase you are running).
2. Paste back to Claude: the **full output of the Run cell**, plus the summary printed by the last cell.
   If anything fails, paste the full traceback.

## 1. Settings

In [ ]:
# ---- edit these ---------------------------------------------------------------
REPO_URL = "https://github.com/<your-username>/flap-routing.git"   # your GitHub repo
BRANCH   = "main"
SCRIPT   = "scripts/p0_hello_gpu.py"
CONFIG   = "configs/p0_hello.yaml"
EXTRA    = "--set require_gpu=true"     # extra args, e.g. "--run-id <id>" to resume a run
PRIVATE_REPO = True   # if True, add a Colab secret named GH_TOKEN (key icon in the left bar):
                      # a fine-grained GitHub token with read-only access to this one repo
# -------------------------------------------------------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/qacr"
REPO_DIR   = "/content/flap-routing"

## 2. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,power.limit --format=csv

## 3. Mount Google Drive (results and checkpoints live here)

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
for sub in ("results", "checkpoints", "hf_cache", "plans"):
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)
os.environ["QACR_DRIVE_ROOT"]   = DRIVE_ROOT
os.environ["QACR_RESULTS_ROOT"] = f"{DRIVE_ROOT}/results"
os.environ["QACR_CKPT_ROOT"]    = f"{DRIVE_ROOT}/checkpoints"
os.environ["QACR_PLANS_DIR"]    = f"{DRIVE_ROOT}/plans"
print("results ->", os.environ["QACR_RESULTS_ROOT"])

## 4. Clone or update the repo

In [ ]:
import os, subprocess
url = REPO_URL
if PRIVATE_REPO:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
    if not token:
        raise RuntimeError("PRIVATE_REPO=True but the Colab secret GH_TOKEN is empty/missing")
    url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")

def git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:  # never print the URL (it may contain the token)
        raise RuntimeError(f"git {args[0]} failed:
{r.stderr.replace(url, '<repo>')}")
    return r.stdout.strip()

if not os.path.isdir(f"{REPO_DIR}/.git"):
    git("clone", "--branch", BRANCH, url, REPO_DIR)
else:
    git("fetch", "origin", BRANCH, cwd=REPO_DIR)
    git("checkout", BRANCH, cwd=REPO_DIR)
    git("reset", "--hard", f"origin/{BRANCH}", cwd=REPO_DIR)
print("commit:", git("log", "-1", "--format=%h %s", cwd=REPO_DIR))
%cd {REPO_DIR}

## 5. Install pinned dependencies (torch: Colab's preinstalled build)

In [ ]:
!pip install -q -r requirements.txt
# Scripts run as fresh subprocesses, so no runtime restart is needed after installing.
!python -c "import torch, transformers; from packaging.version import Version as V; assert V(torch.__version__.split('+')[0]) >= V('2.5'), torch.__version__; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| transformers', transformers.__version__)"

## 6. Unit tests (CPU-only, ~1 min) — must pass before any GPU run

In [ ]:
!PYTEST_DISABLE_PLUGIN_AUTOLOAD=1 python -m pytest -p no:cacheprovider

## 7. Run

In [ ]:
!python {SCRIPT} --config {CONFIG} {EXTRA}

## 7b. (optional) P2 outcome runs for all four variants, one after another

In [ ]:
# Runs only after P1 has produced the plans in MyDrive/qacr/plans/. Each run is resumable:
# if Colab disconnects, re-run this cell and finished tasks are skipped -- but only if you pass
# the same run id, so note the run dirs printed below and use --run-id <id> to resume one.
import subprocess
PLANS = f"{DRIVE_ROOT}/plans"
VARIANTS = [
    "--set variant.name=v0",
    f"--set variant.name=flap_0.50 --set variant.kind=plan --set variant.plan={PLANS}/flap_0.50.safetensors",
    f"--set variant.name=flap_0.20 --set variant.kind=plan --set variant.plan={PLANS}/flap_0.20.safetensors",
    f"--set variant.name=flap_0.35 --set variant.kind=plan --set variant.plan={PLANS}/flap_0.35.safetensors",
]
for v in VARIANTS:
    print("=" * 80, "
", v, flush=True)
    r = subprocess.run(f"python scripts/p2_outcomes.py --config configs/p2_outcomes.yaml --set require_gpu=true {v}",
                       shell=True)
    print("exit code", r.returncode, flush=True)
    if r.returncode != 0:
        break

## 8. Summary of the newest run (paste this back) + download it

In [ ]:
import glob, json, os, shutil
runs = sorted(glob.glob(f"{DRIVE_ROOT}/results/*/*/"), key=os.path.getmtime)
latest = runs[-1].rstrip("/")
print("run dir:", latest)
for name in sorted(os.listdir(latest)):
    print("  ", name, os.path.getsize(f"{latest}/{name}"), "bytes")
summary = [f for f in ("hello.json", "summary.json") if os.path.exists(f"{latest}/{f}")]
for f in summary:
    d = json.load(open(f"{latest}/{f}"))
    def strip(o):
        if isinstance(o, dict): return {k: strip(v) for k, v in o.items() if k != "traceback"}
        return o
    print(json.dumps(strip(d), indent=1)[:6000])
zip_path = shutil.make_archive(f"/content/{os.path.basename(latest)}", "zip", latest)
from google.colab import files
files.download(zip_path)   # put this zip into the repo with: python scripts/import_results.py <zip>